# RT-DETRv3-R18: entrenamiento completo (iteración 1, dataset proxy)

Entrena RT-DETRv3-R18 con la implementación oficial (PaddlePaddle, repositorio `clxia12/RT-DETRv3`) sobre el dataset proxy, con la configuración elegida tras el sondeo del 5 de octubre de 2026: 2 GPU T4, 4 fotos por GPU (lote total 8), tasa de aprendizaje oficial escalada (0.00005), precisión completa, BatchNorm sincronizada entre GPU, 100 épocas, validación en cada época y mejor época por mAP50-95 de validación (el mismo criterio que `best.pt` de YOLO26). Parche versión 2 para los lotes sin cajas.

**100 épocas no caben en una sesión de Kaggle (unas 14.6 h).** Por eso el cuaderno trabaja por sesiones:

1. **Sesión 1** (`REANUDAR = False`): entrena hasta que no quepa otra época antes de `HORAS_MAX` y se detiene justo al terminar una época. Deja en la salida el último punto de control, el mejor modelo de la sesión, los registros y las detecciones de validación del mejor modelo hasta ese momento.
2. **Sesión 2** (`REANUDAR = True`): adjuntar como entrada la salida de la versión de la sesión 1 (*Add Input > Your Work*), lanzar. Continúa desde la última época guardada hasta la 100 y elige el mejor modelo entre las épocas de todas las sesiones.

Configuración del cuaderno: acelerador **GPU T4 x2**, **Internet activado**, dataset `deteccionoso` adjunto. Lanzar con *Save Version > Save & Run All (Commit)*.

Segunda semilla: el mismo cuaderno con `RUN_ID = "e6b_rtdetrv3r18_s1"` y `SEMILLA = 1`; no cambia nada más.

Probado en CPU el 5 de octubre de 2026 con datos sintéticos, dos procesos y 4 épocas: la sesión 1 se detuvo por tiempo al terminar la época 2 y la sesión 2 reanudó desde ahí hasta la 4. No probado en GPU.

## 0. Parámetros

In [ ]:
import time
INICIO_SESION = time.time()      # el límite de Kaggle (12 h) cuenta desde que arranca la sesión

RUN_ID   = "e6_rtdetrv3r18_s0"   # la segunda semilla sería e6b_rtdetrv3r18_s1
SEMILLA  = 0
REANUDAR = False                 # False en la sesión 1; True en las siguientes, con la salida anterior adjunta como entrada

N_GPU        = 2
LOTE_POR_GPU = 4                 # con 8 o 16 por GPU no cabe en una T4
AMP          = False             # precisión mixta: en el sondeo fue 1.10 veces más rápida con 2 GPU
EPOCAS       = 100
HORAS_MAX    = 11.0              # no empieza una época que no termine antes de esta hora de sesión
LR = 0.0004 * LOTE_POR_GPU * N_GPU / 64   # oficial 0.0004 para lote total 64, escalada al lote total

REPO    = "https://github.com/clxia12/RT-DETRv3.git"
COMMIT  = "349e7d99a5065e7b684118912e6a74178d4f4625"     # último commit, verificado el 3 oct 2026
DATASET = "/kaggle/input/datasets/josemartinvallejos/deteccionoso"
ID_DRIVE_PESOS = "1zIDOjn1qDccC3TBsDlGQHOjVrehd26bk"     # RT-DETRv3-R18 COCO (Google Drive del README; el enlace de Baidu da 404)
MD5_PESOS      = "25cdd032d57dbe7773753ae65c300c63"      # medido el 3 oct 2026
print(f"{RUN_ID}: semilla {SEMILLA}, {N_GPU} GPU x {LOTE_POR_GPU} fotos, lr {LR:g}, "
      f"{'precisión mixta' if AMP else 'precisión completa'}, {EPOCAS} épocas, reanudar={REANUDAR}")

## 1. Entorno: Python 3.12 aparte, con Paddle 3.3.0 y las versiones probadas

In [ ]:
import os, sys, subprocess, hashlib
from pathlib import Path

TRABAJO = Path("/kaggle/working")
VENV    = "/tmp/venv_paddle"          # fuera de /kaggle/working: no se guarda con la versión
VENV_PY = f"{VENV}/bin/python"

def correr(cmd, ver=3000, env=None):
    """Corre un comando y muestra el final de su salida y el código de salida."""
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
                       env={**os.environ, **(env or {})})
    if r.stdout.strip():
        print(r.stdout[-ver:])
    if r.stderr.strip():
        print("[stderr]", r.stderr[-ver:])
    print("-> código de salida:", r.returncode, "\n")
    return r

REQUISITOS = """
anyio==4.15.1 babel==2.18.0 bce-python-sdk==0.9.79 blinker==1.9.0 certifi==2026.7.22
charset-normalizer==3.5.2 click==8.5.0 cloudpickle==3.1.2 contourpy==1.3.3 crc32c==2.9.post0
cycler==0.12.1 cython==3.3.0 flask==3.1.3 flask-babel==4.0.0 fonttools==4.66.1 future==1.0.0
h11==0.16.0 httpcore==1.0.9 httpx==0.28.1 idna==3.20 imageio==2.38.0 imgaug==0.4.0
itsdangerous==2.2.0 jinja2==3.1.6 joblib==1.6.0 kiwisolver==1.5.1 lapx==0.10.0 lazy-loader==0.6
markupsafe==3.0.4 matplotlib==3.11.2 motmetrics==1.4.0 narwhals==2.26.0 networkx==3.7
numpy==1.26.4 opencv-python==4.5.5.64 opt-einsum==3.3.0 packaging==26.3 pandas==3.0.6
pillow==12.3.0 protobuf==7.36.2 psutil==7.2.2 pyclipper==1.4.0 pycocotools==2.0.11
pycryptodome==3.23.0 pyparsing==3.3.3 python-dateutil==2.9.0.post0 pytz==2026.5 pyyaml==6.0.3
rarfile==4.5 requests==2.34.2 safetensors==0.8.0 scikit-image==0.26.0 scikit-learn==1.9.1
scipy==1.17.1 setuptools==80.10.2 shapely==2.1.2 six==1.17.0 terminaltables==3.1.10
threadpoolctl==3.7.0 tifffile==2026.3.3 tqdm==4.70.1 typeguard==4.6.0 typing-extensions==4.16.0
urllib3==2.8.0 visualdl==2.5.3 werkzeug==3.1.9 xmltodict==1.0.4
""".split()
(TRABAJO / "requisitos_rtdetrv3.txt").write_text("\n".join(REQUISITOS) + "\n")
print(len(REQUISITOS), "paquetes fijados")   # 67

correr("nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv")
print("CPU que ve la sesión:", os.cpu_count())
correr([sys.executable, "-m", "pip", "install", "-q", "uv"])
correr([sys.executable, "-m", "uv", "python", "install", "3.12.13"])
correr([sys.executable, "-m", "uv", "venv", "--seed", "--clear", "--python", "3.12.13", VENV])
correr([VENV_PY, "-m", "pip", "install", "-q", "-r", str(TRABAJO / "requisitos_rtdetrv3.txt")])
correr([VENV_PY, "-m", "pip", "install", "-q", "paddlepaddle-gpu==3.3.0",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        "--extra-index-url", "https://pypi.org/simple"])
correr([VENV_PY, "-m", "pip", "check"])
correr([VENV_PY, "-c", "import paddle; paddle.utils.run_check()"], env={"CUDA_VISIBLE_DEVICES": "0,1"})

## 2. Repositorio oficial, pesos COCO y parche

In [ ]:
REPO_DIR = Path("/tmp/RT-DETRv3")      # fuera de /kaggle/working: no hace falta guardarlo
if not REPO_DIR.exists():
    correr(["git", "clone", "-q", REPO, str(REPO_DIR)])
correr(["git", "-C", str(REPO_DIR), "checkout", "-q", COMMIT])
correr(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %ci %s"])

PESOS_COCO = Path("/tmp/pesos/rtdetrv3_r18vd_6x_coco.pdparams")
PESOS_COCO.parent.mkdir(exist_ok=True)
if not PESOS_COCO.exists():
    correr([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    correr(["gdown", ID_DRIVE_PESOS, "-O", str(PESOS_COCO)])
md5 = hashlib.md5(PESOS_COCO.read_bytes()).hexdigest()
print(PESOS_COCO.name, f"{PESOS_COCO.stat().st_size / 1e6:.1f} MB", "md5", md5)
assert md5 == MD5_PESOS, "Los pesos descargados no son los esperados"

In [ ]:
%%writefile /kaggle/working/parche_lotes_sin_cajas_v2.py
"""Parche del proyecto oso andino (versión 2, 2026-10-05) al código oficial de RT-DETRv3.

Problema: si ninguna imagen de un lote de entrenamiento tiene cajas, la función de denoising devuelve None
para todos los grupos de consultas y el código de RT-DETRv3 falla al unir piezas (paddle.concat con None).
El RT-DETR original sí contempla ese caso (comprueba None); la versión 3, con varios grupos, no.

Este parche hace que esos lotes se entrenen como en el RT-DETR original: sin consultas de denoising, solo con
las consultas normales, cuya pérdida de clasificación enseña al modelo que en esas fotos no hay oso.
Reemplaza al parche de la versión 1, que saltaba esos lotes (y que con 2 GPU congelaría el entrenamiento).

Toca tres sitios:
  1. rtdetr_transformerv3.py, _get_decoder_input: no concatena el denoising de un grupo si es None.
  2. rtdetr_transformerv3.py, forward: arma la máscara de atención de un grupo sin denoising y le pone un
     dn_meta de marcador con el número de consultas del grupo, para que la cabeza pueda separar los grupos.
  3. detr_head.py, DINOv3Head.forward: en los grupos marcados calcula la pérdida sin denoising y repite la
     reducción entre GPUs que la pérdida de denoising habría hecho, para que todas las GPUs hagan las mismas.

Uso: python parche_lotes_sin_cajas_v2.py RUTA_DEL_REPOSITORIO   (se puede correr varias veces)"""
import sys
from pathlib import Path

repo = Path(sys.argv[1])
MARCA = "Parche del proyecto oso andino v2"

CAMBIOS = {
    repo / "ppdet/modeling/transformers/rtdetr_transformerv3.py": [
        (  # 1a. referencia de las cajas de denoising
            "            if denoising_bbox_unacts is not None and not (self.o2m_branch and g_id == self.num_groups - 1):\n",
            "            if denoising_bbox_unacts is not None and not (self.o2m_branch and g_id == self.num_groups - 1) \\\n"
            "                    and denoising_bbox_unacts[g_id] is not None:  # " + MARCA + "\n",
        ),
        (  # 1b. consultas de denoising
            "            if denoising_classes is not None and not (self.o2m_branch and g_id == self.num_groups - 1):\n",
            "            if denoising_classes is not None and not (self.o2m_branch and g_id == self.num_groups - 1) \\\n"
            "                    and denoising_classes[g_id] is not None:  # " + MARCA + "\n",
        ),
        (  # 2a. máscara de atención de un grupo sin denoising
            "                else:\n"
            "                    end = end + attn_masks[g_id].shape[1]\n",
            "                elif attn_masks[g_id] is None:  # " + MARCA + ": lote sin cajas, grupo sin denoising\n"
            "                    end = end + self.num_queries[g_id]\n"
            "                    new_mask = new_mask > 0.1 if g_id > 0 else new_mask >= 0.0\n"
            "                    new_attn_mask[begin: end, begin: end] = new_mask\n"
            "                else:\n"
            "                    end = end + attn_masks[g_id].shape[1]\n",
        ),
        (  # 2b. marcador para la cabeza
            "            attn_masks = new_attn_mask\n",
            "            attn_masks = new_attn_mask\n"
            "            # " + MARCA + ": los grupos sin denoising llevan un dn_meta de marcador con su número de consultas\n"
            "            dn_metas = [dm if dm is not None else {'dn_num_split': [0, self.num_queries[i]], 'sin_denoising': True}\n"
            "                        for i, dm in enumerate(dn_metas)]\n",
        ),
    ],
    repo / "ppdet/modeling/heads/detr_head.py": [
        (  # 3a. separación por grupo
            "                for g_id in range(num_groups):\n"
            "                    dn_out_bboxes_gid, dec_out_bboxes_gid = paddle.split(\n"
            "                        dec_out_bboxes[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n"
            "                    dn_out_logits_gid, dec_out_logits_gid = paddle.split(\n"
            "                        dec_out_logits[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n",
            "                for g_id in range(num_groups):\n"
            "                    sin_dn = dn_meta[g_id].get('sin_denoising', False)  # " + MARCA + "\n"
            "                    if sin_dn:\n"
            "                        dn_out_bboxes_gid = dn_out_logits_gid = None\n"
            "                        dec_out_bboxes_gid, dec_out_logits_gid = dec_out_bboxes[g_id], dec_out_logits[g_id]\n"
            "                    else:\n"
            "                        dn_out_bboxes_gid, dec_out_bboxes_gid = paddle.split(\n"
            "                            dec_out_bboxes[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n"
            "                        dn_out_logits_gid, dec_out_logits_gid = paddle.split(\n"
            "                            dec_out_logits[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n",
        ),
        (  # 3b. pérdida sin denoising y reducción entre GPUs equivalente
            "                        dn_out_logits=dn_out_logits_gid,\n"
            "                        dn_meta=dn_meta[g_id])\n",
            "                        dn_out_logits=dn_out_logits_gid,\n"
            "                        dn_meta=None if sin_dn else dn_meta[g_id])\n"
            "                    if sin_dn:  # " + MARCA + ": misma cantidad de reducciones entre GPUs que con denoising\n"
            "                        self.loss._get_num_gts(inputs['gt_class'])\n",
        ),
    ],
}

for ruta, cambios in CAMBIOS.items():
    texto = ruta.read_text()
    if MARCA in texto:
        print("Ya estaba aplicado en", ruta.name)
        continue
    for viejo, nuevo in cambios:
        n = texto.count(viejo)
        if n != 1:
            raise SystemExit(f"{ruta.name}: el texto de anclaje aparece {n} veces (se esperaba 1); no se parcha nada.")
        texto = texto.replace(viejo, nuevo)
    ruta.write_text(texto)
    print("Parche aplicado en", ruta.name)

In [ ]:
%%writefile /kaggle/working/entrenar_con_semilla.py
"""Lanza tools/train.py del repositorio oficial de RT-DETRv3 fijando antes la semilla.
El repositorio solo permite la semilla 0 (opción --enable_ce); esto llama a la misma función
(ppdet.engine.set_random_seed) con la semilla que se elija, sin modificar el código oficial.
Uso: python entrenar_con_semilla.py RUTA_REPO SEMILLA [argumentos de tools/train.py ...]"""
import runpy
import sys

repo, semilla, resto = sys.argv[1], int(sys.argv[2]), sys.argv[3:]
if "--enable_ce" in resto:
    raise SystemExit("No uses --enable_ce aquí: volvería a fijar la semilla en 0.")
sys.path.insert(0, repo)
from ppdet.engine import set_random_seed
set_random_seed(semilla)
print(f"Semilla fijada en {semilla} (paddle, random y numpy)", flush=True)
sys.argv = [f"{repo}/tools/train.py"] + resto
runpy.run_path(f"{repo}/tools/train.py", run_name="__main__")

In [ ]:
correr([VENV_PY, str(TRABAJO / "parche_lotes_sin_cajas_v2.py"), str(REPO_DIR)])
correr(["git", "-C", str(REPO_DIR), "diff", "--stat"])

## 3. Anotaciones COCO y configuración

In [ ]:
import json
from PIL import Image

COCO_DIR = Path("/tmp/coco")
GIRA = {5, 6, 7, 8}   # orientaciones EXIF con las que OpenCV gira la foto e intercambia ancho y alto

def yolo_a_coco(dir_imagenes, dir_etiquetas, salida_json, nombre_clase="oso"):
    """Etiquetas YOLO de una clase -> JSON COCO. Entran todas las imágenes; las negativas, sin anotaciones."""
    imagenes, anotaciones = [], []
    n_pos = n_neg = n_recortadas = n_descartadas = n_exif = 0
    for img_id, ruta in enumerate(sorted(Path(dir_imagenes).glob("*.jpg")), start=1):
        with Image.open(ruta) as im:
            W, H = im.size                      # solo lee la cabecera
            orient = im.getexif().get(0x0112, 1)
        if orient in GIRA:                      # PaddleDetection (cv2.imdecode) y Ultralytics (cv2.imread) aplican el giro
            W, H = H, W
        n_exif += orient != 1
        imagenes.append({"id": img_id, "file_name": ruta.name, "width": W, "height": H})
        txt = Path(dir_etiquetas) / (ruta.stem + ".txt")
        filas = [l.split() for l in txt.read_text().splitlines() if l.strip()] if txt.exists() else []
        cajas_img = 0
        for f in filas:
            cx, cy, w, h = (float(v) for v in f[1:5])
            x1, y1, x2, y2 = (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H
            r = (max(0.0, x1), max(0.0, y1), min(float(W), x2), min(float(H), y2))
            n_recortadas += r != (x1, y1, x2, y2)
            x1, y1, x2, y2 = r
            if x2 - x1 <= 0 or y2 - y1 <= 0:
                n_descartadas += 1
                continue
            anotaciones.append({"id": len(anotaciones) + 1, "image_id": img_id, "category_id": 1,
                                "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1), "iscrowd": 0})
            cajas_img += 1
        n_pos += cajas_img > 0
        n_neg += cajas_img == 0
    Path(salida_json).parent.mkdir(parents=True, exist_ok=True)
    Path(salida_json).write_text(json.dumps({"images": imagenes, "annotations": anotaciones,
                                             "categories": [{"id": 1, "name": nombre_clase}]}))
    return {"imagenes": len(imagenes), "con_oso": n_pos, "sin_oso": n_neg, "cajas": len(anotaciones),
            "cajas_recortadas_al_borde": n_recortadas, "cajas_descartadas": n_descartadas,
            "con_orientacion_exif": n_exif}

ESPERADO = {"train": {"imagenes": 2656, "con_oso": 1382, "cajas": 1507},   # informe de preparación
            "val":   {"imagenes": 598,  "con_oso": 326,  "cajas": 351}}
for conj in ("train", "val"):
    res = yolo_a_coco(Path(DATASET) / "images" / conj, Path(DATASET) / "labels" / conj, COCO_DIR / f"{conj}.json")
    print(conj, res)
    for k, v in ESPERADO[conj].items():
        assert res[k] == v, f"{conj}: {k} = {res[k]}, se esperaba {v}"
print("Conteos iguales a los del informe de preparación.\n")

CONFIG = REPO_DIR / "configs" / "rtdetrv3" / "rtdetrv3_r18vd_oso.yml"
CONFIG.write_text(f"""# RT-DETRv3-R18 para la detección de oso (iteración 1, dataset proxy). Hereda la receta oficial.
_BASE_: [
  'rtdetrv3_r18vd_6x_coco.yml',
]

num_classes: 1
metric: COCO

TrainDataset:
  name: COCODataSet
  dataset_dir: {DATASET}
  image_dir: images/train
  anno_path: {COCO_DIR / 'train.json'}
  data_fields: ['image', 'gt_bbox', 'gt_class', 'is_crowd']
  allow_empty: true            # sin esto se descartan las fotos sin oso

EvalDataset:
  name: COCODataSet
  dataset_dir: {DATASET}
  image_dir: images/val
  anno_path: {COCO_DIR / 'val.json'}
  allow_empty: true

TestDataset:
  name: ImageFolder
  anno_path: {COCO_DIR / 'val.json'}

epoch: 100                     # protocolo común con YOLO26 (la receta oficial usa 72)
LearningRate:
  base_lr: {LR:.8f}            # oficial: 0.0004 para lote total 64; escalada al lote total
TrainReader:
  batch_size: {LOTE_POR_GPU}                # por GPU; con 8 y 16 no cabe en una T4 de 15 GB
  use_shared_memory: false     # con memoria compartida, las fotos sin oso tumban el lector de datos (Paddle 3.3.0)
pretrain_weights: {PESOS_COCO}
""")
print(CONFIG.read_text())

## 4. Carpeta de la corrida y reanudación

Todo lo que se guarda va a `/kaggle/working/<RUN_ID>/`. Con `REANUDAR = True` busca en `/kaggle/input` la carpeta `<RUN_ID>` que dejó la sesión anterior, copia su historial (mejores modelos, registros, métricas) y reanuda desde su último punto de control. El optimizador, la tasa de aprendizaje y el promedio EMA continúan donde quedaron (`ppdet/utils/checkpoint.py`, `load_weight`).

In [ ]:
import json, shutil, glob

CORRIDA = TRABAJO / RUN_ID
ENTRENO = CORRIDA / "entreno"            # save_dir de PaddleDetection durante la sesión
ENTRENO.mkdir(parents=True, exist_ok=True)
shutil.copy(CONFIG, TRABAJO / CONFIG.name)

REANUDAR_DESDE, EPOCAS_PREVIAS = None, 0
if REANUDAR:
    candidatos = []
    for p in glob.glob(f"/kaggle/input/**/{RUN_ID}/ultimo/ultimo.json", recursive=True):
        info = json.loads(Path(p).read_text())
        candidatos.append((info["epocas_completas"], Path(p).parent.parent))
    assert candidatos, f"No hay en /kaggle/input una carpeta {RUN_ID}/ultimo: adjuntar la salida de la sesión anterior."
    EPOCAS_PREVIAS, ANTERIOR = max(candidatos, key=lambda c: c[0])
    print(f"Sesión anterior encontrada en {ANTERIOR}: {EPOCAS_PREVIAS} épocas completas")
    for p in ANTERIOR.iterdir():         # historial: mejores modelos, registros y métricas de las sesiones previas
        if p.name not in ("ultimo", "entreno"):
            (shutil.copytree if p.is_dir() else shutil.copy)(p, CORRIDA / p.name)
    REANUDAR_DESDE = ANTERIOR / "ultimo" / "ultimo"
    assert EPOCAS_PREVIAS < EPOCAS, "La corrida ya tiene todas sus épocas."
SESION = 1 + len(list(CORRIDA.glob("train_s*.log")))
print(f"Sesión {SESION} de {RUN_ID}; reanuda desde: {REANUDAR_DESDE or 'los pesos COCO'}")

## 5. Entrenamiento

Vigila el registro cada 30 segundos. Al empezar cada época borra los puntos de control de las épocas anteriores (se conservan el último y el mejor) y decide si cabe otra época antes de `HORAS_MAX`; si no cabe, detiene el entrenamiento ahí, con la época anterior ya guardada y evaluada. Se detiene también si aparece un NaN en la pérdida.

In [ ]:
import re, signal
from datetime import datetime

LANZADOR = TRABAJO / "entrenar_con_semilla.py"
opciones = (["--amp"] if AMP else []) + (["-r", str(REANUDAR_DESDE)] if REANUDAR_DESDE else []) + \
           ["-c", str(CONFIG), "--eval", "-o", f"epoch={EPOCAS}", f"save_dir={ENTRENO}", "log_iter=50"]
dispositivos = ",".join(str(i) for i in range(N_GPU))
if N_GPU == 1:
    cmd = [VENV_PY, str(LANZADOR), str(REPO_DIR), str(SEMILLA), *opciones]
    REGISTRO = CORRIDA / f"consola_s{SESION}.log"
else:
    cmd = [VENV_PY, "-m", "paddle.distributed.launch", "--devices", dispositivos, "--log_dir", str(ENTRENO / "log"),
           str(LANZADOR), str(REPO_DIR), str(SEMILLA), *opciones]
    REGISTRO = ENTRENO / "log" / "workerlog.0"
print("$", " ".join(cmd), "\n", flush=True)

LIMITE = INICIO_SESION + HORAS_MAX * 3600
LIMITE_DURO = LIMITE + 0.4 * 3600           # por si una época se alarga: corta aunque sea a mitad de época
patron_ckpt = re.compile(r"^(\d+)\.(pdparams|pdema|pdopt|pdstates)$")
patron_inicio = re.compile(r"\[(\d\d/\d\d \d\d:\d\d:\d\d)\] ppdet\.engine\.callbacks INFO: Epoch: \[(\d+)\] \[\s*0/")
inicio, mem, ultimo_aviso, motivo = time.time(), {}, time.time(), None
detenido_por_tiempo, epoca_vista = False, None

def leer_registro():
    return REGISTRO.read_text(errors="replace") if REGISTRO.exists() else ""

def inicios_de_epoca(texto):
    """Épocas que ya empezaron y la hora de su primer paso (el año se fija para poder restar fechas)."""
    return [(int(e), datetime.strptime("2026/" + t, "%Y/%m/%d %H:%M:%S")) for t, e in patron_inicio.findall(texto)]

with open(CORRIDA / f"consola_s{SESION}.log", "w") as f:
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
                            env={**os.environ, "CUDA_VISIBLE_DEVICES": dispositivos, "MPLBACKEND": "Agg"})
    while proc.poll() is None:
        time.sleep(30)
        q = subprocess.run("nvidia-smi --query-gpu=index,memory.used --format=csv,noheader,nounits",
                           shell=True, capture_output=True, text=True)
        if q.returncode == 0 and q.stdout.strip():
            for l in q.stdout.strip().splitlines():
                k, v = (int(x) for x in l.split(","))
                mem[k] = max(mem.get(k, 0), v)
        texto = leer_registro()
        if re.search(r" loss: nan", texto, re.IGNORECASE):
            motivo = "NaN en la pérdida"
            break
        inicios = inicios_de_epoca(texto)
        if inicios:
            actual = inicios[-1][0]                 # la época en curso; las anteriores ya están guardadas y evaluadas
            for p in ENTRENO.iterdir():             # se conserva solo el punto de control de la última época completa
                m = patron_ckpt.match(p.name)
                if m and int(m.group(1)) < actual - 1:
                    p.unlink()
            if actual != epoca_vista:              # acaba de empezar una época: ¿terminará antes del límite?
                epoca_vista = actual
                if len(inicios) >= 2:
                    dur = (inicios[-1][1] - inicios[-2][1]).total_seconds()
                    if time.time() + 1.05 * dur > LIMITE:
                        motivo, detenido_por_tiempo = f"no cabe otra época antes de {HORAS_MAX:g} h de sesión", True
                        break
        if time.time() > LIMITE_DURO:
            motivo, detenido_por_tiempo = "límite duro de tiempo (a mitad de época)", True
            break
        if time.time() - ultimo_aviso >= 900:      # un aviso de avance cada 15 minutos
            lineas = [l for l in texto.splitlines() if "Epoch: [" in l]
            m = re.search(r"Epoch: \[(\d+)\] \[\s*(\d+)/(\d+)\].* loss: (\S+).* eta: (\S+)", lineas[-1]) if lineas else None
            if m:
                print(f"  {(time.time() - INICIO_SESION) / 3600:5.2f} h de sesión | época {m.group(1)} paso {m.group(2)}/{m.group(3)} "
                      f"| pérdida {m.group(4)} | tiempo restante de la corrida según PaddleDetection {m.group(5)}", flush=True)
            ultimo_aviso = time.time()
    if proc.poll() is None:                         # se detiene el entrenamiento (y sus procesos hijos)
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=120)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
    elif motivo is None:
        motivo = "terminó todas las épocas" if proc.returncode == 0 else f"falló con código {proc.returncode}"

texto = leer_registro()
shutil.copy(REGISTRO, CORRIDA / f"train_s{SESION}.log") if REGISTRO.exists() else None
print(f"\nMotivo de fin: {motivo} | {(time.time() - inicio) / 3600:.2f} h de entrenamiento en esta sesión | "
      f"memoria máx: {mem} MiB")
if not detenido_por_tiempo and motivo != "terminó todas las épocas":    # al detenerlo, Paddle avisa de la señal de parada
    consola = (CORRIDA / f"consola_s{SESION}.log").read_text(errors="replace")
    for linea in (texto + consola).splitlines():
        if any(k in linea for k in ("Traceback", "Error", "out of memory", "Out of memory")):
            print(linea[:200])

## 6. Puntos de control, métricas por época y mejor modelo

- `ultimo/`: el punto de control de la última época completa, para la sesión siguiente.
- `mejor_s<N>/`: el mejor modelo de esta sesión y su métrica (`best_model.pdstates`).
- `<RUN_ID>_epocas.csv`: mAP50-95, mAP50 y AR de validación por época (de PaddleDetection), y la duración de cada época.
- `<RUN_ID>_best.pdparams` y `<RUN_ID>_best.json`: el mejor modelo entre todas las sesiones hasta ahora.

In [ ]:
# 1) Último punto de control completo: el de mayor época con sus cuatro archivos (o model_final)
completos = {}
for p in ENTRENO.iterdir():
    m = re.match(r"^(\d+|model_final)\.(pdparams|pdema|pdopt|pdstates)$", p.name)
    if m:
        completos.setdefault(m.group(1), set()).add(m.group(2))
completos = {k: v for k, v in completos.items() if v >= {"pdparams", "pdema", "pdopt", "pdstates"}}
inicios = inicios_de_epoca(texto)
ultima_iniciada = inicios[-1][0] if inicios else -1
if "model_final" in completos:
    nombre_ultimo, epocas_completas = "model_final", EPOCAS
elif completos:
    n = max(int(k) for k in completos)
    nombre_ultimo, epocas_completas = str(n), n + 1
else:
    nombre_ultimo, epocas_completas = None, EPOCAS_PREVIAS
if nombre_ultimo:
    destino = CORRIDA / "ultimo"
    shutil.rmtree(destino, ignore_errors=True)
    destino.mkdir()
    for ext in ("pdparams", "pdema", "pdopt", "pdstates"):
        shutil.move(str(ENTRENO / f"{nombre_ultimo}.{ext}"), destino / f"ultimo.{ext}")
    (destino / "ultimo.json").write_text(json.dumps({"run_id": RUN_ID, "epocas_completas": epocas_completas,
                                                     "sesion": SESION}))
print(f"Épocas completas: {epocas_completas} de {EPOCAS} (punto de control: {nombre_ultimo})")

# 2) Mejor modelo de esta sesión
if (ENTRENO / "best_model.pdparams").exists():
    mejor = CORRIDA / f"mejor_s{SESION}"
    shutil.rmtree(mejor, ignore_errors=True)
    mejor.mkdir()
    for ext in ("pdparams", "pdstates"):
        shutil.copy(ENTRENO / f"best_model.{ext}", mejor / f"best_model.{ext}")

# 3) Métricas de validación por época, leídas del registro (todas las sesiones)
def metricas_por_epoca(texto, sesion):
    filas, epoca = {}, None
    for linea in texto.splitlines():
        m = re.search(r"Epoch: \[(\d+)\] \[", linea)
        if m:
            epoca = int(m.group(1))
            continue
        if epoca is None:
            continue
        v = re.search(r"\] = ([-\d.]+)\s*$", linea)
        if v and "area=   all | maxDets=100" in linea:
            f = filas.setdefault(epoca, {"epoca": epoca + 1, "sesion": sesion})
            if "Average Precision" in linea and "IoU=0.50:0.95" in linea:
                f["mAP50_95"] = float(v.group(1))
            elif "Average Precision" in linea and "IoU=0.50 " in linea:
                f["mAP50"] = float(v.group(1))
            elif "Average Recall" in linea and "IoU=0.50:0.95" in linea:
                f["AR100"] = float(v.group(1))
    ini = inicios_de_epoca(texto)
    for (e1, t1), (e2, t2) in zip(ini, ini[1:]):
        if e1 in filas:
            filas[e1]["min_epoca"] = round((t2 - t1).total_seconds() / 60, 1)
    return list(filas.values())

import pandas as pd
filas = []
for log in sorted(CORRIDA.glob("train_s*.log")):
    filas += metricas_por_epoca(log.read_text(errors="replace"), int(re.search(r"_s(\d+)\.log$", log.name).group(1)))
EPOCAS_DF = pd.DataFrame(filas).sort_values("epoca").drop_duplicates("epoca", keep="last")
EPOCAS_DF.to_csv(TRABAJO / f"{RUN_ID}_epocas.csv", index=False)
print(EPOCAS_DF.tail(10).to_string(index=False))

# 4) Mejor modelo entre todas las sesiones (mAP50-95 de validación, como best.pt de YOLO26)
r = subprocess.run([VENV_PY, "-c", "import json, sys, paddle; print(json.dumps([{'metric': float(d['metric']), "
                    "'epoch': int(d['epoch']), 'ruta': p} for p in sys.argv[1:] for d in [paddle.load(p)]]))",
                    *map(str, sorted(CORRIDA.glob("mejor_s*/best_model.pdstates")))],
                   capture_output=True, text=True)
mejores = json.loads(r.stdout.strip().splitlines()[-1]) if r.returncode == 0 else []
for m in mejores:
    print(f"   {Path(m['ruta']).parent.name}: época {m['epoch']}, mAP50-95 {m['metric']:.4f}")
if mejores:
    BEST = max(mejores, key=lambda m: (m["metric"], m["epoch"]))   # empate: la época más tardía, como best.pt y PaddleDetection
    shutil.copy(Path(BEST["ruta"]).with_suffix(".pdparams"), TRABAJO / f"{RUN_ID}_best.pdparams")
    (TRABAJO / f"{RUN_ID}_best.json").write_text(json.dumps(
        {"run_id": RUN_ID, "epoca": BEST["epoch"], "mAP50_95_paddle": BEST["metric"],
         "de": Path(BEST["ruta"]).parent.name, "epocas_completas": epocas_completas}, indent=1))
    print(f"Mejor hasta ahora: época {BEST['epoch']} (mAP50-95 {BEST['metric']:.4f}), de {Path(BEST['ruta']).parent.name}")

## 7. Detecciones de validación del mejor modelo

En el formato que lee el cuaderno de evaluación (`archivo, conf, x1, y1, x2, y2` normalizadas, confianza ≥ 0.001). La evaluación de PaddleDetection reescala cada foto a 640 × 640 sin mantener la proporción, igual al procesarlas en lotes o una por una.

In [ ]:
if (TRABAJO / f"{RUN_ID}_best.pdparams").exists():
    EVAL_DIR = Path("/tmp/eval_val")
    shutil.rmtree(EVAL_DIR, ignore_errors=True)
    r = correr([VENV_PY, str(REPO_DIR / "tools/eval.py"), "-c", str(CONFIG), "--output_eval", str(EVAL_DIR),
                "-o", f"weights={TRABAJO / (RUN_ID + '_best')}"], ver=1500,
               env={"CUDA_VISIBLE_DEVICES": "0", "MPLBACKEND": "Agg"})
    g = json.loads((COCO_DIR / "val.json").read_text())
    info = {im["id"]: (im["file_name"], im["width"], im["height"]) for im in g["images"]}
    filas = []
    for d in json.loads((EVAL_DIR / "bbox.json").read_text()):
        nombre, W, H = info[d["image_id"]]
        x, y, w, h = d["bbox"]
        filas.append((nombre, d["score"], max(0.0, x) / W, max(0.0, y) / H, min(W, x + w) / W, min(H, y + h) / H))
    det = pd.DataFrame(filas, columns=["archivo", "conf", "x1", "y1", "x2", "y2"])
    det = det[det.conf >= 0.001]
    det.round(6).to_csv(TRABAJO / f"{RUN_ID}_detecciones_val.csv.gz", index=False)
    print(f"{len(det)} detecciones en {det.archivo.nunique()} de {len(info)} imágenes -> {RUN_ID}_detecciones_val.csv.gz")

## 8. Registro del entorno y limpieza

In [ ]:
def salida(cmd):
    return subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True).stdout

(CORRIDA / f"entorno_s{SESION}.txt").write_text("\n\n".join([
    salida([VENV_PY, "-m", "pip", "freeze"]), salida("nvidia-smi"), salida(["git", "-C", str(REPO_DIR), "diff"]),
]))
shutil.rmtree(ENTRENO, ignore_errors=True)        # lo útil ya está en ultimo/ y mejor_s*/
for p in sorted(TRABAJO.rglob("*")):
    if p.is_file():
        print(f"   {p.relative_to(TRABAJO)}  {p.stat().st_size / 1e6:.1f} MB")
faltan = EPOCAS - epocas_completas
print(f"\n{RUN_ID}: {epocas_completas} de {EPOCAS} épocas. Motivo de fin de esta sesión: {motivo}.")
if faltan > 0 and motivo and "NaN" not in motivo and "falló" not in motivo:
    print(f"Faltan {faltan} épocas: lanzar otra sesión con REANUDAR = True y esta versión adjunta como entrada.")